In [1]:
import os
import pandas as pd
import requests
from tqdm import tqdm

# Change working directory after imports
os.chdir(r'C:\Users\hamod4691\OneDrive - UiT Office 365\Data paper code\CB')

# Get missing AphiaIDs

In [2]:
# Function to query WoRMS API for AphiaID and additional metadata
def get_aphia_info(taxa_list):
    results = []
    
    # Iterate through the taxa list with a progress bar
    for taxon in tqdm(taxa_list, desc="Querying WoRMS"):
        try:
            # Query the WoRMS API (marine_only=true ensures only marine taxa are included)
            url = f"https://www.marinespecies.org/rest/AphiaRecordsByName/{taxon}?marine_only=true"
            response = requests.get(url)
            
            if response.status_code == 200:
                records = response.json()
                if records:  # If records are found
                    record = records[0]  # Take only the first match
                    results.append({
                        "AphiaID": record.get("AphiaID"),
                        "url": record.get("url"),
                        "scientificname": record.get("scientificname"),
                        "authority": record.get("authority"),
                        "taxonRankID": record.get("taxonRankID"),
                        "rank": record.get("rank"),
                        "status": record.get("status"),
                        "unacceptreason": record.get("unacceptreason"),
                        "valid_AphiaID": record.get("valid_AphiaID"),
                        "valid_name": record.get("valid_name"),
                        "valid_authority": record.get("valid_authority"),
                        "parentNameUsageID": record.get("parentNameUsageID"),
                        "kingdom": record.get("kingdom"),
                        "phylum": record.get("phylum"),
                        "class": record.get("class"),
                        "order": record.get("order"),
                        "family": record.get("family"),
                        "genus": record.get("genus"),
                        "citation": record.get("citation"),
                        "lsid": record.get("lsid"),
                        "isMarine": record.get("isMarine"),
                        "isBrackish": record.get("isBrackish"),
                        "isFreshwater": record.get("isFreshwater"),
                        "isTerrestrial": record.get("isTerrestrial"),
                        "isExtinct": record.get("isExtinct"),
                        "match_type": record.get("match_type"),
                        "modified": record.get("modified"),
                        "Input_Name": taxon  # Add the input name for reference
                    })
                else:
                    # No records found
                    results.append({"Input_Name": taxon})
            else:
                # API error
                results.append({"Input_Name": taxon})
        except Exception as e:
            # Handle any exceptions
            print(f"Error querying taxon '{taxon}': {e}")
            results.append({"Input_Name": taxon})
    
    return results
# Load taxa list from the 'unique' sheet in the Excel file
taxa_file = "missing_aphiaid.xlsx"  
taxa_data = pd.read_excel(taxa_file, sheet_name="unique")  
# Inspect column names and extract the taxa list
print(taxa_data.columns)  # Inspect column names
taxa_list = taxa_data["ScientificName"]  # Replace with the correct column name
# Query WoRMS for each taxon
results = get_aphia_info(taxa_list)
# Convert results to a DataFrame
results_df = pd.DataFrame(results)
# Save the results to a new CSV file
results_df.to_csv("missing_aphiaid_updated.csv", index=False)
# Print the results
print(results_df)

Index(['AphiaID', 'ScientificName', 'Taxon Status', 'AphiaID (Accepted)',
       'ScientificName (Accepted)', 'ScientificName (User)'],
      dtype='object')


Querying WoRMS: 100%|██████████| 1005/1005 [11:55<00:00,  1.41it/s]

       AphiaID                                                url  \
0     101010.0  https://www.marinespecies.org/aphia.php?p=taxd...   
1     101061.0  https://www.marinespecies.org/aphia.php?p=taxd...   
2     101078.0  https://www.marinespecies.org/aphia.php?p=taxd...   
3     101170.0  https://www.marinespecies.org/aphia.php?p=taxd...   
4     101383.0  https://www.marinespecies.org/aphia.php?p=taxd...   
...        ...                                                ...   
1000     960.0  https://www.marinespecies.org/aphia.php?p=taxd...   
1001  988323.0  https://www.marinespecies.org/aphia.php?p=taxd...   
1002  989667.0  https://www.marinespecies.org/aphia.php?p=taxd...   
1003  992877.0  https://www.marinespecies.org/aphia.php?p=taxd...   
1004  994868.0  https://www.marinespecies.org/aphia.php?p=taxd...   

              scientificname                  authority  taxonRankID     rank  \
0          Cerianthus lloydi                Gosse, 1859        220.0  Species   
1        

In [3]:
# Load the original Excel file (Biota sheet)
original_file = "original.xlsx"
biota_df = pd.read_excel(original_file, sheet_name="Biota")
# Load the CSV file with missing AphiaID information
missing_file = "missing_aphiaid_updated.csv"
missing_df = pd.read_csv(missing_file)
# Inspect the column names in both files
print("Biota columns:", biota_df.columns)
print("Missing AphiaID columns:", missing_df.columns)
# Merge the two datasets using ScientificName (Biota) and Input_Name (missing CSV) as the key
merged_df = biota_df.merge(
    missing_df,
    how="left",  # Perform a left join to keep all rows from the Biota sheet
    left_on="ScientificName",  # Key column in Biota sheet
    right_on="Input_Name"  # Key column in the missing CSV
)
# Fill in the empty cells in the specified columns
merged_df["Taxon Status"] = merged_df["Taxon Status"].combine_first(merged_df["status"])
merged_df["AphiaId (Accepted)"] = merged_df["AphiaId (Accepted)"].combine_first(merged_df["AphiaID"])
merged_df["ScientificName (Accepted)"] = merged_df["ScientificName (Accepted)"].combine_first(merged_df["scientificname"])
# Drop unnecessary columns from the merge
merged_df = merged_df.drop(columns=["status", "AphiaID", "scientificname", "Input_Name"])
# Save the updated DataFrame back to the original Excel file
output_file = "original_biota.xlsx"
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:  # Removed invalid mode argument
    merged_df.to_excel(writer, sheet_name="Biota", index=False)
print(f"Updated file saved to: {output_file}")

Biota columns: Index(['CruiseID', 'StationID', 'DeviceID', 'SampleID', 'SubSampleID',
       'SubsetID', 'BiotaID', 'Cruise', 'Station', 'Device', 'Sample',
       'SubSample', 'Subset', 'Biota', 'AphiaId', 'ScientificName',
       'Taxon Status', 'AphiaId (Accepted)', 'ScientificName (Accepted)',
       'ScientificName (User)', 'Count', 'Abundance', 'PresenceAbsence',
       'SumWet', 'SumDry', 'SumAFDW', 'Stage', 'SieveFraction',
       'SubTotalFactor', 'Specification', 'Remark'],
      dtype='object')
Missing AphiaID columns: Index(['AphiaID', 'url', 'scientificname', 'authority', 'taxonRankID', 'rank',
       'status', 'unacceptreason', 'valid_AphiaID', 'valid_name',
       'valid_authority', 'parentNameUsageID', 'kingdom', 'phylum', 'class',
       'order', 'family', 'genus', 'citation', 'lsid', 'isMarine',
       'isBrackish', 'isFreshwater', 'isTerrestrial', 'isExtinct',
       'match_type', 'modified', 'Input_Name'],
      dtype='object')
Updated file saved to: original_biota.

In [4]:
# Load the original Excel file (Taxon sheet)
original_file = "original.xlsx"
taxon_df = pd.read_excel(original_file, sheet_name="Taxon")
# Load the CSV file with missing AphiaID information
missing_file = "missing_aphiaid_updated.csv"
missing_df = pd.read_csv(missing_file)
# Inspect the column names in both files
print("Taxon columns:", taxon_df.columns)
print("Missing AphiaID columns:", missing_df.columns)
# Merge the two datasets using ScientificName (Taxon) and Input_Name (missing CSV) as the key
merged_df = taxon_df.merge(
    missing_df,
    how="left",  # Perform a left join to keep all rows from the Taxon sheet
    left_on="ScientificName",  # Key column in Taxon sheet
    right_on="Input_Name"  # Key column in the missing CSV
)
# Fill in the empty cells in the specified columns
columns_to_fill = {
    "ValidScientificName": "valid_name",
    "AcceptedAphiaId": "AphiaID",
    "AcceptedScientificName": "scientificname",
    "Status": "status",
    "UnacceptReason": "unacceptreason",
    "Rank": "rank",
    "Kingdom": "kingdom",
    "Phylum": "phylum",
    "Class": "class",
    "Order": "order",
    "Family": "family",
    "Genus": "genus",
    "ValidAphiaId": "valid_AphiaID",
    "ValidAuthority": "valid_authority",
    "Citation": "citation",
    "URL": "url",
    "LSID": "lsid",
    "IsMarine": "isMarine",
    "IsBrackish": "isBrackish",
    "IsFreshwater": "isFreshwater",
    "IsTerrestrial": "isTerrestrial",
    "IsExtinct": "isExtinct",
    "Authority": "authority",
    "MatchType": "match_type",
    "Modified": "modified"
}
# Iterate through the columns and fill empty cells
for original_col, missing_col in columns_to_fill.items():
    if original_col in merged_df.columns and missing_col in merged_df.columns:
        merged_df[original_col] = merged_df[original_col].combine_first(merged_df[missing_col])
# Drop unnecessary columns from the merge
merged_df = merged_df.drop(columns=missing_df.columns.difference(["Input_Name"]))
# Save the updated DataFrame back to the original Excel file
output_file = "original_taxon.xlsx"
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:  # Removed invalid mode argument
    merged_df.to_excel(writer, sheet_name="Taxon", index=False)
print(f"Updated file saved to: {output_file}")

Taxon columns: Index(['AphiaId', 'ScientificName', 'ValidScientificName', 'AcceptedAphiaId',
       'AcceptedScientificName', 'Status', 'UnacceptReason', 'Rank', 'Kingdom',
       'Phylum', 'Class', 'Order', 'Family', 'Genus', 'ValidAphiaId',
       'ValidAuthority', 'Citation', 'URL', 'LSID', 'IsMarine', 'IsBrackish',
       'IsFreshwater', 'IsTerrestrial', 'IsExtinct', 'IsColony', 'Authority',
       'MatchType', 'Modified'],
      dtype='object')
Missing AphiaID columns: Index(['AphiaID', 'url', 'scientificname', 'authority', 'taxonRankID', 'rank',
       'status', 'unacceptreason', 'valid_AphiaID', 'valid_name',
       'valid_authority', 'parentNameUsageID', 'kingdom', 'phylum', 'class',
       'order', 'family', 'genus', 'citation', 'lsid', 'isMarine',
       'isBrackish', 'isFreshwater', 'isTerrestrial', 'isExtinct',
       'match_type', 'modified', 'Input_Name'],
      dtype='object')
Updated file saved to: original_taxon.xlsx


Go through and remove extinct/terrestrial? 

# Find GBIF ID for taxa in CB

In [5]:
# Load the Excel file and read the Taxon sheet
file_path = "original_taxon.xlsx"  
taxa_df = pd.read_excel(file_path, sheet_name="Taxon")
# Function to query GBIF API for a taxon
def query_gbif(scientific_name, authority=None):
    # Construct the API query
    url = f"https://api.gbif.org/v1/species/match?name={scientific_name}"
    if authority:
        url += f"&authorship={authority}"
    
    try:
        response = requests.get(url)
        if response.status_code == 200:
            result = response.json()
            if "usageKey" in result:
                return {
                    "TaxonKey": result.get("usageKey"),
                    "MatchedName": result.get("scientificName"),
                    "Rank": result.get("rank"),
                    "Status": result.get("status"),
                    "Confidence": result.get("confidence"),
                    "Alternatives": result.get("alternatives", [])
                }
            else:
                return {"TaxonKey": None, "MatchedName": None, "Rank": None, "Status": None, "Confidence": None, "Alternatives": []}
        else:
            print(f"Failed to fetch data for {scientific_name}. HTTP Status: {response.status_code}")
            return {"TaxonKey": None, "MatchedName": None, "Rank": None, "Status": None, "Confidence": None, "Alternatives": []}
    except Exception as e:
        print(f"Error fetching data for {scientific_name}: {e}")
        return {"TaxonKey": None, "MatchedName": None, "Rank": None, "Status": None, "Confidence": None, "Alternatives": []}
# Query GBIF for each taxon in the DataFrame
results = []
for _, row in tqdm(taxa_df.iterrows(), total=len(taxa_df), desc="Querying GBIF"):
    scientific_name = row["AcceptedScientificName"]
    authority = row.get("ValidAuthority", None)  # Use authority if available
    result = query_gbif(scientific_name, authority)
    result["AcceptedScientificName"] = scientific_name
    result["ValidAuthority"] = authority
    results.append(result)
# Convert results to a DataFrame
results_df = pd.DataFrame(results)
# Save the results to a CSV file
output_file = "gbif_taxonkey_original.csv"
results_df.to_csv(output_file, index=False)
print(f"Results saved to {output_file}")

Querying GBIF: 100%|██████████| 5584/5584 [40:32<00:00,  2.30it/s]

Results saved to gbif_taxonkey_original.csv


In [6]:
# Manual fix for kinetoskias
# Load the CSV file into a DataFrame
file_path = "gbif_taxonkey_original.csv"  # Replace with the correct file path
df = pd.read_csv(file_path)
# Update the row where AcceptedScientificName is 'Kinetoskias'
df.loc[df["AcceptedScientificName"] == "Kinetoskias", ["TaxonKey", "MatchedName", "Rank", "Status"]] = [
    102141373,  # TaxonKey
    "Kinetoskias",  # MatchedName
    "Genus",  # Rank
    "ACCEPTED"  # Status
]
# Save the updated DataFrame back to the CSV file
output_file = "gbif_taxonkey_original.csv"  # Specify the output file name
df.to_csv(output_file, index=False)
print(f"Updated file saved to: {output_file}")

Updated file saved to: gbif_taxonkey_original.csv
